# Databricks notebook source

### EDA — Features de Red SCADA
#### **Dataset:** Ventanas temporales de 1 segundo, etiquetadas como Normal (0) o Ataque (1)
####
#### **Bloques:**
#### 1. Estructura y balance de clases
#### 2. Distribución temporal de ataques
#### 3. Volumen de tráfico (general y por label)
#### 4. Protocolo y aplicaciones
#### 5. SCADA Tags y Modbus
#### 6. Duración de conexiones y gaps
#### 7. Correlaciones y separabilidad

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

DELTA_FINAL_PATH = "/Volumes/workspace/default/network_data/features_delta/"

df = spark.read.format("delta").load(DELTA_FINAL_PATH)

print(f"Total ventanas : {df.count():,}")
print(f"Columnas       : {len(df.columns)}")
print(f"Schema:")
df.printSchema()

## 1 — Estructura y balance de clases

In [0]:
total = df.count()

dist = (
    df.groupBy("label")
      .count()
      .withColumn("pct", F.round(F.col("count") / total * 100, 2))
      .orderBy("label")
      .toPandas()
)

print(dist.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Barras
colors = ["#4C8BF5", "#E8453C"]
axes[0].bar(["Normal (0)", "Ataque (1)"], dist["count"], color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Distribución de clases", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Ventanas")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, row in dist.iterrows():
    axes[0].text(i, row["count"] + 2000, f"{row['pct']}%", ha="center", fontsize=11)

# Pie
axes[1].pie(dist["count"], labels=["Normal (0)", "Ataque (1)"],
            colors=colors, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Proporción Normal / Ataque", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 2 — Distribución temporal de ataques

In [0]:
df_time = df.withColumn("hora", F.date_trunc("hour", F.col("window_start")))

timeline = (
    df_time.groupBy("hora", "label")
           .count()
           .orderBy("hora")
           .toPandas()
)

normal  = timeline[timeline["label"] == 0]
ataque  = timeline[timeline["label"] == 1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(normal["hora"], normal["count"], alpha=0.5, color="#4C8BF5", label="Normal")
ax.fill_between(ataque["hora"], ataque["count"], alpha=0.7, color="#E8453C", label="Ataque")
ax.set_title("Ventanas por hora — Normal vs Ataque", fontsize=13, fontweight="bold")
ax.set_xlabel("Fecha / Hora")
ax.set_ylabel("Nº ventanas")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3 — Volumen de tráfico

In [0]:
# Columnas numéricas de volumen disponibles — ajusta si tu schema difiere
vol_cols = ["total_packets", "outbound_packets", "inbound_packets",
            "tcp_packets", "udp_packets"]

# Estadísticas generales
display(df.select(vol_cols).describe())

In [0]:
# Distribución general de total_packets (histograma)
pdf_vol = df.select("total_packets", "label").sample(fraction=0.3, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma general
axes[0].hist(pdf_vol["total_packets"], bins=60, color="#4C8BF5", edgecolor="white", linewidth=0.5)
axes[0].set_title("Distribución total_packets (muestra 30%)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("total_packets")
axes[0].set_ylabel("Frecuencia")

# Boxplot Normal vs Ataque
bp_data = [
    pdf_vol[pdf_vol["label"] == 0]["total_packets"].values,
    pdf_vol[pdf_vol["label"] == 1]["total_packets"].values,
]
bp = axes[1].boxplot(bp_data, labels=["Normal (0)", "Ataque (1)"],
                     patch_artist=True, notch=True)
for patch, color in zip(bp["boxes"], ["#4C8BF5", "#E8453C"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("total_packets — Normal vs Ataque", fontsize=12, fontweight="bold")
axes[1].set_ylabel("total_packets")

plt.tight_layout()
plt.show()

In [0]:
pdf_vol = df.select("total_packets", "label").sample(fraction=0.3, seed=42).toPandas()

normal = pdf_vol[pdf_vol["label"] == 0]["total_packets"]
ataque = pdf_vol[pdf_vol["label"] == 1]["total_packets"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma separado por clase
axes[0].hist(normal, bins=60, color="#4C8BF5", alpha=0.7, edgecolor="white", linewidth=0.5, label="Normal (0)")
axes[0].hist(ataque, bins=60, color="#E8453C", alpha=0.7, edgecolor="white", linewidth=0.5, label="Ataque (1)")
axes[0].set_title("Distribución total_packets por clase (muestra 30%)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("total_packets")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Boxplot Normal vs Ataque
bp = axes[1].boxplot(
    [normal.values, ataque.values],
    labels=["Normal (0)", "Ataque (1)"],
    patch_artist=True,
    notch=True
)
for patch, color in zip(bp["boxes"], ["#4C8BF5", "#E8453C"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("total_packets — Normal vs Ataque", fontsize=12, fontweight="bold")
axes[1].set_ylabel("total_packets")

plt.tight_layout()
plt.show()

In [0]:
# Medias de todas las métricas de volumen por label
vol_stats = (
    df.groupBy("label")
      .agg(*[F.round(F.mean(c), 3).alias(f"mean_{c}") for c in vol_cols])
      .orderBy("label")
      .toPandas()
)

print("Media de métricas de volumen por clase:")
print(vol_stats.to_string(index=False))

# Visualización comparativa
vol_means = vol_stats.set_index("label").T
vol_means.columns = ["Normal", "Ataque"]
vol_means.index = [c.replace("mean_", "") for c in vol_means.index]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(vol_means))
width = 0.35
ax.bar(x - width/2, vol_means["Normal"], width, label="Normal", color="#4C8BF5", alpha=0.85)
ax.bar(x + width/2, vol_means["Ataque"],  width, label="Ataque",  color="#E8453C", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(vol_means.index, rotation=15)
ax.set_title("Media de métricas de volumen — Normal vs Ataque", fontsize=12, fontweight="bold")
ax.set_ylabel("Media de paquetes")
ax.legend()
plt.tight_layout()
plt.show()

## 4 — Protocolo y aplicaciones

In [0]:
# Proporción TCP vs UDP por label
proto_stats = (
    df.groupBy("label")
      .agg(
          F.round(F.mean("tcp_packets"), 2).alias("mean_tcp"),
          F.round(F.mean("udp_packets"), 2).alias("mean_udp"),
      )
      .orderBy("label")
      .toPandas()
)
print(proto_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (label_val, label_name, color) in enumerate([(0, "Normal", "#4C8BF5"), (1, "Ataque", "#E8453C")]):
    row = proto_stats[proto_stats["label"] == label_val].iloc[0]
    axes[i].pie(
        [row["mean_tcp"], row["mean_udp"]],
        labels=["TCP", "UDP"],
        colors=["#2563EB", "#F59E0B"],
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"edgecolor": "white"}
    )
    axes[i].set_title(f"Protocolo — {label_name}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## SCADA Tags y Modbus

In [0]:
# Distribución general del tag más activo por ventana
# (asumiendo que tienes columnas tipo cnt_tag_HMI_LIT101, etc.)
# Si tienes una columna "dominant_tag" o similar, úsala directamente.
# Si no, mostramos los conteos de los tags como features numéricas.

tag_cols = [c for c in df.columns if "tag" in c.lower() or "HMI" in c]
print("Columnas relacionadas con tags encontradas:")
print(tag_cols)

In [0]:
# Conteos medios de cada tag por label
if tag_cols:
    tag_stats = (
        df.groupBy("label")
          .agg(*[F.round(F.mean(c), 3).alias(f"mean_{c}") for c in tag_cols])
          .orderBy("label")
          .toPandas()
    )

    tag_means = tag_stats.set_index("label").T
    tag_means.columns = ["Normal", "Ataque"]
    tag_means.index = [c.replace("mean_", "") for c in tag_means.index]

    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(tag_means))
    width = 0.35
    ax.bar(x - width/2, tag_means["Normal"], width, label="Normal", color="#4C8BF5", alpha=0.85)
    ax.bar(x + width/2, tag_means["Ataque"],  width, label="Ataque",  color="#E8453C", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(tag_means.index, rotation=30, ha="right")
    ax.set_title("Media de accesos por SCADA Tag — Normal vs Ataque", fontsize=12, fontweight="bold")
    ax.set_ylabel("Media de accesos")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron columnas de SCADA tags. Revisa el nombre exacto en tu schema.")

In [0]:
# Función Modbus — distribución de read vs write por label
modbus_cols = [c for c in df.columns if "read" in c.lower() or "write" in c.lower() or "modbus" in c.lower()]
print("Columnas Modbus encontradas:", modbus_cols)

if modbus_cols:
    modbus_stats = (
        df.groupBy("label")
          .agg(*[F.round(F.mean(c), 3).alias(f"mean_{c}") for c in modbus_cols])
          .orderBy("label")
          .toPandas()
    )
    display(modbus_stats)

## 6 — Duración de conexiones y gaps temporales

In [0]:
duration_cols = [c for c in df.columns if "duration" in c.lower() or "gap" in c.lower() or "conn" in c.lower()]
print("Columnas de duración/gap encontradas:", duration_cols)

In [0]:
if duration_cols:
    # Estadísticas por label
    agg_exprs = []
    for c in duration_cols:
        agg_exprs.append(F.round(F.mean(c),   3).alias(f"mean_{c}"))
        agg_exprs.append(F.round(F.stddev(c), 3).alias(f"std_{c}"))

    dur_stats = (
        df.groupBy("label")
          .agg(*agg_exprs)
          .orderBy("label")
    )
    display(dur_stats)

    # Boxplot de la columna más representativa
    main_dur = duration_cols[0]
    pdf_dur = df.select(main_dur, "label").filter(F.col(main_dur).isNotNull()) \
                .sample(fraction=0.3, seed=42).toPandas()

    fig, ax = plt.subplots(figsize=(7, 4))
    bp_data = [
        pdf_dur[pdf_dur["label"] == 0][main_dur].values,
        pdf_dur[pdf_dur["label"] == 1][main_dur].values,
    ]
    bp = ax.boxplot(bp_data, labels=["Normal (0)", "Ataque (1)"],
                    patch_artist=True, notch=True)
    for patch, color in zip(bp["boxes"], ["#4C8BF5", "#E8453C"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f"{main_dur} — Normal vs Ataque", fontsize=12, fontweight="bold")
    ax.set_ylabel(main_dur)
    plt.tight_layout()
    plt.show()

## 7 

In [0]:
# Seleccionar columnas numéricas (excluyendo IDs y timestamps)
exclude = ["window_id", "label", "session_id", "window_start", "window_end",
           "gap_before_seconds", "is_after_gap", "prev_window_id"]

num_cols = [
    c for c in df.columns
    if df.schema[c].dataType.simpleString() in ("double", "long", "int", "float")
    and c not in exclude
]

print(f"Features numéricas para correlación: {len(num_cols)}")
print(num_cols)

In [0]:
# Muestra para correlación (máx 50k filas, suficiente)
pdf_corr = df.select(num_cols + ["label"]).sample(fraction=0.1, seed=42).toPandas()

corr_matrix = pdf_corr[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(num_cols, fontsize=8)
ax.set_title("Matriz de correlación — features numéricas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
# Correlación de cada feature con el label — ranking de importancia lineal
corr_label = (
    pdf_corr[num_cols + ["label"]]
    .corr()["label"]
    .drop("label")
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
corr_label.columns = ["feature", "corr_abs_con_label"]
print("Top 15 features más correladas con el label:")
print(corr_label.head(15).to_string(index=False))

top15 = corr_label.head(15)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top15["feature"][::-1], top15["corr_abs_con_label"][::-1],
               color="#4C8BF5", edgecolor="white")
ax.set_title("Top 15 features — correlación absoluta con label", fontsize=12, fontweight="bold")
ax.set_xlabel("Correlación absoluta")
ax.axvline(0.1, color="gray", linestyle="--", linewidth=0.8, label="umbral 0.1")
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Añadir columnas temporales
df_time = (
    spark.read.format("delta").load("/Volumes/workspace/default/network_data/features_fe/")
    .withColumn("hora",         F.hour("window_start"))
    .withColumn("dia_semana",   F.dayofweek("window_start"))  # 1=Dom, 2=Lun, ..., 7=Sab
    .withColumn("dia_nombre",   F.date_format("window_start", "EEEE"))
    .withColumn("franja",
        F.when((F.col("hora") >= 6)  & (F.col("hora") < 12), "Mañana (6-12)")
         .when((F.col("hora") >= 12) & (F.col("hora") < 20), "Tarde (12-20)")
         .otherwise("Noche (20-6)")
    )
)

# Traer a pandas
pdf_time = df_time.select("hora", "dia_semana", "dia_nombre", "franja", "label").toPandas()
pdf_time["label"] = pdf_time["label"].astype(int)

print(f"Total ventanas: {len(pdf_time):,}")

In [0]:
# ── Orden correcto de días
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nombres_es  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
dia_map     = dict(zip(orden_dias, nombres_es))

pdf_time["dia_es"] = pdf_time["dia_nombre"].map(dia_map)

fig, axes = plt.subplots(3, 1, figsize=(13, 14))

# ── 1. Por día de la semana
dia_stats = (
    pdf_time.groupby(["dia_es", "label"])
            .size().reset_index(name="count")
)
dia_pivot = dia_stats.pivot(index="dia_es", columns="label", values="count").fillna(0)
dia_pivot.columns = ["Normal", "Ataque"]
dia_pivot = dia_pivot.reindex(nombres_es)

x = np.arange(len(dia_pivot))
w = 0.35
axes[0].bar(x - w/2, dia_pivot["Normal"], w, label="Normal",
            color="#4C8BF5", alpha=0.85)
axes[0].bar(x + w/2, dia_pivot["Ataque"], w, label="Ataque",
            color="#E8453C", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(dia_pivot.index, fontsize=11)
axes[0].set_title("Distribución por día de la semana", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Ventanas")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Añadir % de ataques sobre cada día
for i, dia in enumerate(dia_pivot.index):
    total_dia = dia_pivot.loc[dia, "Normal"] + dia_pivot.loc[dia, "Ataque"]
    pct = dia_pivot.loc[dia, "Ataque"] / total_dia * 100 if total_dia > 0 else 0
    axes[0].text(i + w/2, dia_pivot.loc[dia, "Ataque"] + 200,
                 f"{pct:.1f}%", ha="center", fontsize=9, color="#E8453C")

# ── 2. Por franja horaria
orden_franja = ["Mañana (6-12)", "Tarde (12-20)", "Noche (20-6)"]
franja_stats = (
    pdf_time.groupby(["franja", "label"])
            .size().reset_index(name="count")
)
franja_pivot = franja_stats.pivot(index="franja", columns="label", values="count").fillna(0)
franja_pivot.columns = ["Normal", "Ataque"]
franja_pivot = franja_pivot.reindex(orden_franja)

x2 = np.arange(len(franja_pivot))
axes[1].bar(x2 - w/2, franja_pivot["Normal"], w, label="Normal",
            color="#4C8BF5", alpha=0.85)
axes[1].bar(x2 + w/2, franja_pivot["Ataque"], w, label="Ataque",
            color="#E8453C", alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(franja_pivot.index, fontsize=11)
axes[1].set_title("Distribución por franja horaria", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Ventanas")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

for i, franja in enumerate(franja_pivot.index):
    total_f = franja_pivot.loc[franja, "Normal"] + franja_pivot.loc[franja, "Ataque"]
    pct = franja_pivot.loc[franja, "Ataque"] / total_f * 100 if total_f > 0 else 0
    axes[1].text(i + w/2, franja_pivot.loc[franja, "Ataque"] + 200,
                 f"{pct:.1f}%", ha="center", fontsize=10, color="#E8453C")

# ── 3. Por hora del día
hora_stats = (
    pdf_time.groupby(["hora", "label"])
            .size().reset_index(name="count")
)
hora_pivot = hora_stats.pivot(index="hora", columns="label", values="count").fillna(0)
hora_pivot.columns = ["Normal", "Ataque"]
hora_pivot = hora_pivot.reindex(range(24)).fillna(0)

x3 = np.arange(24)
axes[2].bar(x3 - w/2, hora_pivot["Normal"], w, label="Normal",
            color="#4C8BF5", alpha=0.85)
axes[2].bar(x3 + w/2, hora_pivot["Ataque"], w, label="Ataque",
            color="#E8453C", alpha=0.85)
axes[2].set_xticks(x3)
axes[2].set_xticklabels([f"{h:02d}h" for h in range(24)], rotation=45, fontsize=9)
axes[2].set_title("Distribución por hora del día", fontsize=13, fontweight="bold")
axes[2].set_ylabel("Ventanas")
axes[2].legend()
axes[2].grid(axis="y", alpha=0.3)

plt.suptitle("Análisis temporal de ataques", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [0]:
# ── Tabla resumen con % de ataques por hora
hora_pct = hora_pivot.copy()
hora_pct["total"]   = hora_pct["Normal"] + hora_pct["Ataque"]
hora_pct["pct_ataque"] = (hora_pct["Ataque"] / hora_pct["total"] * 100).round(2)
hora_pct.index.name = "hora"
hora_pct = hora_pct.reset_index()

print("% de ataques por hora:")
print(hora_pct[["hora", "Normal", "Ataque", "total", "pct_ataque"]].to_string(index=False))

# Hora con más ataques en proporción
top_hora = hora_pct.loc[hora_pct["pct_ataque"].idxmax()]
print(f"\nHora con mayor proporción de ataques: {int(top_hora['hora']):02d}h ({top_hora['pct_ataque']}%)")

## Resumen final

In [0]:
print("=" * 55)
print("RESUMEN EDA")
print("=" * 55)
print(f"Total ventanas          : {total:,}")

dist_dict = dict(zip(dist["label"], dist["count"]))
print(f"  Normal (0)            : {dist_dict.get(0, 0):,}  ({dist[dist['label']==0]['pct'].values[0]}%)")
print(f"  Ataque (1)            : {dist_dict.get(1, 0):,}  ({dist[dist['label']==1]['pct'].values[0]}%)")
print(f"Features numéricas      : {len(num_cols)}")
print(f"Top feature (corr label): {corr_label.iloc[0]['feature']}  ({corr_label.iloc[0]['corr_abs_con_label']:.3f})")
print("=" * 55)